In [1]:
# 1. Install Dependencies
!pip install diffusers transformers accelerate datasets gradio

# 2. Import Libraries
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from diffusers import StableDiffusionPipeline, DDPMScheduler
from datasets import load_dataset
import gradio as gr
from transformers import CLIPTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 140.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Observed the butterflies dataset:

In [3]:
# 3. Load and Preprocess Dataset

dataset_raw = load_dataset("ceyda/smithsonian_butterflies", split="train")
print(f"Loaded {len(dataset_raw)} images")
print(dataset_raw[0].keys())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

train-00000-of-00004.parquet:   0%|          | 0.00/496M [00:00<?, ?B/s]

train-00001-of-00004.parquet:   0%|          | 0.00/451M [00:00<?, ?B/s]

train-00002-of-00004.parquet:   0%|          | 0.00/476M [00:00<?, ?B/s]

train-00003-of-00004.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9213 [00:00<?, ? examples/s]

Loaded 9213 images
dict_keys(['image_url', 'image_alt', 'id', 'name', 'scientific_name', 'gender', 'taxonomy', 'region', 'locality', 'date', 'usnm_no', 'guid', 'edan_url', 'source', 'stage', 'image', 'image_hash', 'sim_score'])


## Clip Tokenizer

In [4]:

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")


tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

# Define the ButterflyDataset

In [5]:
from torch.utils.data import DataLoader
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, DDPMScheduler
import torchvision.transforms as T
class ButterflyHFDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, tokenizer, image_size=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
            T.CenterCrop(image_size),
            T.RandomHorizontalFlip(),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        entry = self.dataset[idx]
        image = entry['image'].convert("RGB")

        # Initialize the combo way of caption
        name = entry.get("name", "butterfly")
        sci = entry.get("scientific_name", "")
        region = entry.get("region", "")
        stage = entry.get("stage", "")
        caption = f"A {stage} butterfly named {name} ({sci}) from {region}."

        image = self.transform(image)
        inputs = self.tokenizer(caption, return_tensors="pt", padding="max_length", truncation=True, max_length=77)

        return {
            "pixel_values": image,
            "input_ids": inputs.input_ids.squeeze(0)
        }

# DataLoader

In [6]:
# Dataloader
dataset = ButterflyHFDataset(dataset_raw, tokenizer)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"Loaded {len(dataset)} images")
print(dataset[0].keys())

Loaded 9213 images
dict_keys(['pixel_values', 'input_ids'])


# Customize UNet Structure

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, context_dim):
        super().__init__()
        self.query = nn.Conv2d(dim, dim, 1)
        self.key = nn.Linear(context_dim, dim)
        self.value = nn.Linear(context_dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, context):
        B, C, H, W = x.shape
        q = self.query(x).flatten(2).transpose(1, 2)   # (B, HW, C)
        k = self.key(context)                          # (B, 77, C)
        v = self.value(context)                        # (B, 77, C)

        attn = self.softmax(torch.bmm(q, k.transpose(1, 2)) / (C ** 0.5))  # (B, HW, 77)
        out = torch.bmm(attn, v).transpose(1, 2).reshape(B, C, H, W)       # (B, C, H, W)
        return out

class PromptEmbedding(nn.Module):
    def __init__(self, context_dim, n_tokens=1):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(n_tokens, context_dim))

    def forward(self, batch_size):
        return self.embedding.unsqueeze(0).expand(batch_size, -1, -1)  # (B, n_tokens, dim)

class CustomUNetWithAttention(nn.Module):
    def __init__(self, latent_channels=4, text_embed_dim=512, use_prompt_tuning=True):
        super().__init__()
        self.use_prompt_tuning = use_prompt_tuning
        if use_prompt_tuning:
            self.learned_prompt = PromptEmbedding(text_embed_dim, n_tokens=1)

        # Encoder
        self.down1 = nn.Conv2d(latent_channels, 64, kernel_size=3, padding=1)
        self.down2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.down_res = nn.Conv2d(128, 128, 3, padding=1)

        # Cross Attention
        self.cross_attn = CrossAttentionBlock(128, text_embed_dim)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=3, padding=1)
        self.final = nn.Conv2d(64, latent_channels, kernel_size=3, padding=1)

    def forward(self, x, timesteps, encoder_hidden_states):
        if self.use_prompt_tuning:
            B = x.shape[0]
            learned_tokens = self.learned_prompt(B)
            encoder_hidden_states = torch.cat([encoder_hidden_states, learned_tokens], dim=1)  # (B, 78, dim)

        x = F.relu(self.down1(x))
        x = F.relu(self.down2(x))
        x = F.relu(self.down_res(x))

        x = x + self.cross_attn(x, encoder_hidden_states)  # Add cross-attention output

        x = F.relu(self.up1(x))
        x = self.final(x)

        return type("UNetOutput", (), {"sample": x})


# Load the model
###Scheduler + optimizer

In [8]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, DDPMScheduler


# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load models
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)
unet = CustomUNetWithAttention(use_prompt_tuning=True).to(device)
scheduler = DDPMScheduler.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="scheduler")
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, unet.parameters()), lr=1e-4)

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

# Training

In [9]:
from tqdm import tqdm

# Training loop
for epoch in range(20):
    loop = tqdm(dataloader, desc=f"Epoch {epoch}")
    for batch in loop:
        images = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * 0.18215
            text_embeds = text_encoder(input_ids).last_hidden_state

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, scheduler.num_train_timesteps, (latents.shape[0],), device=device).long()
        noisy_latents = scheduler.add_noise(latents, noise, timesteps)

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=text_embeds)
        loss = F.mse_loss(noise_pred.sample, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())



Epoch 0:   0%|          | 0/2304 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/diffusers/configuration_utils.py:140: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
Epoch 19: 100%|██████████| 2304/2304 [08:15<00:00,  4.65it/s, loss=0.0148]


In [10]:
SAVE_PATH = "/content/drive/MyDrive/new-stable-diffusion-checkpoints"

import os
os.makedirs(SAVE_PATH, exist_ok=True)

# Save Fine-Tuned Model
torch.save(unet.state_dict(), os.path.join(SAVE_PATH, f"unet_epoch_{epoch}.pt"))

In [11]:
unet = CustomUNetWithAttention(
    latent_channels=4,
    text_embed_dim=512,
    use_prompt_tuning=True
).to(device)

checkpoint_path = "/content/drive/MyDrive/new-stable-diffusion-checkpoints/unet_epoch_20.pt"
unet.load_state_dict(torch.load(checkpoint_path, map_location=device))
unet.eval()


CustomUNetWithAttention(
  (learned_prompt): PromptEmbedding()
  (down1): Conv2d(4, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (down_res): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (cross_attn): CrossAttentionBlock(
    (query): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1))
    (key): Linear(in_features=512, out_features=128, bias=True)
    (value): Linear(in_features=512, out_features=128, bias=True)
    (softmax): Softmax(dim=-1)
  )
  (up1): ConvTranspose2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (final): Conv2d(64, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

In [12]:
from transformers import CLIPTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, DDPMScheduler

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)
scheduler = DDPMScheduler.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="scheduler")


In [18]:
# Save the full model pipeline in Diffusers format
import os
import json
from shutil import copyfile

FULL_MODEL_PATH = "/content/drive/MyDrive/custom-unet_stable_diffusion"
os.makedirs(FULL_MODEL_PATH, exist_ok=True)

# Create subdirectories
subdirs = ["feature_extractor", "safety_checker", "scheduler", "text_encoder", "tokenizer", "unet", "vae"]
for subdir in subdirs:
    os.makedirs(os.path.join(FULL_MODEL_PATH, subdir), exist_ok=True)

# Save UNet
torch.save(unet.state_dict(), os.path.join(FULL_MODEL_PATH, "unet", "diffusion_pytorch_model.bin"))
with open(os.path.join(FULL_MODEL_PATH, "unet", "config.json"), "w") as f:
    json.dump({
        "sample_size": 64,
        "in_channels": 4,
        "out_channels": 4,
        "layers_per_block": 2,
        "cross_attention_dim": 512,
        "use_prompt_tuning": True
    }, f, indent=2)

# Save tokenizer configuration
tokenizer.save_pretrained(os.path.join(FULL_MODEL_PATH, "tokenizer"))

# Save text encoder configuration
text_encoder.save_pretrained(os.path.join(FULL_MODEL_PATH, "text_encoder"))

# Save VAE configuration
vae.save_pretrained(os.path.join(FULL_MODEL_PATH, "vae"))

# Save scheduler configuration
scheduler.save_pretrained(os.path.join(FULL_MODEL_PATH, "scheduler"))

# Create empty config files for components we're not using
with open(os.path.join(FULL_MODEL_PATH, "feature_extractor", "config.json"), "w") as f:
    json.dump({"feature_extractor_type": "CLIPFeatureExtractor"}, f, indent=2)

with open(os.path.join(FULL_MODEL_PATH, "safety_checker", "config.json"), "w") as f:
    json.dump({"safety_checker_type": "StableDiffusionSafetyChecker"}, f, indent=2)

# Create model index file
model_index = {
    "format": "custom_butterfly_sd",
    "_class_name": "StableDiffusionPipeline",
    "_diffusers_version": "0.16.1",
    "feature_extractor": ["feature_extractor"],
    "safety_checker": ["safety_checker"],
    "scheduler": ["scheduler"],
    "text_encoder": ["text_encoder"],
    "tokenizer": ["tokenizer"],
    "unet": ["unet"],
    "vae": ["vae"]
}

with open(os.path.join(FULL_MODEL_PATH, "model_index.json"), "w") as f:
    json.dump(model_index, f, indent=2)

print(f"Full model saved in Diffusers format to {FULL_MODEL_PATH}")

Full model saved in Diffusers format to /content/drive/MyDrive/custom-unet_stable_diffusion


In [ ]:
import gradio as gr
from PIL import Image
import numpy as np

def generate(prompt):
    with torch.no_grad():
        input_ids = tokenizer(prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=77).input_ids.to(device)
        text_embeds = text_encoder(input_ids).last_hidden_state

        latents = torch.randn(1, 4, 64, 64).to(device)
        for t in scheduler.timesteps:
            latent_model_input = scheduler.scale_model_input(latents, t)
            noise_pred = unet(latent_model_input, torch.tensor([t]).to(device), encoder_hidden_states=text_embeds).sample
            latents = scheduler.step(noise_pred, t, latents).prev_sample

        image = vae.decode(latents / 0.18215).sample
        image = (image.clamp(-1, 1) + 1) / 2
        image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
        return Image.fromarray((image * 255).astype(np.uint8))

gr.Interface(fn=generate, inputs="text", outputs="image", title="Butterfly Generator 🦋").launch(debug=True)


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ad36f2094f97438fb1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
